In [0]:
%sql
CREATE CATALOG IF NOT EXISTS energy_catalog;

In [0]:
%sql
USE CATALOG energy_catalog;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS raw;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS logs; 

In [0]:
%sql
USE CATALOG energy_catalog;
USE SCHEMA raw;

In [0]:
%sql
-- Energy Usage
CREATE TABLE IF NOT EXISTS energy_catalog.raw.src_energy_usage
USING PARQUET
LOCATION 's3://energy-pipline-prod/parquet/energy_usage/'
OPTIONS (
  mergeSchema = "true"
);

In [0]:
%sql
-- Weather
CREATE TABLE IF NOT EXISTS energy_catalog.raw.src_weather
USING PARQUET
LOCATION 's3://energy-pipline-prod/parquet/weather/'
OPTIONS (
  mergeSchema = "true"
);

In [0]:
%sql
-- Device Metrics
CREATE TABLE IF NOT EXISTS energy_catalog.raw.src_device_metrics
USING PARQUET
LOCATION 's3://energy-pipline-prod/parquet/device_metrics/'
OPTIONS (
  mergeSchema = "true"
);

In [0]:
%sql
-- Grid Load
CREATE TABLE IF NOT EXISTS energy_catalog.raw.src_grid_load
USING PARQUET
LOCATION 's3://energy-pipline-prod/parquet/grid_load/'
OPTIONS (
  mergeSchema = "true"
);

In [0]:
%sql
-- Tariff Metrics
CREATE TABLE IF NOT EXISTS energy_catalog.raw.src_tariff_metrics
USING PARQUET
LOCATION 's3://energy-pipline-prod/parquet/tariff_metrics/'
OPTIONS (
  mergeSchema = "true"
);

In [0]:
%sql
SHOW TABLES IN energy_catalog.raw;

In [0]:
%sql
SELECT 'energy_usage'   AS stream, COUNT(*) AS rows FROM energy_catalog.raw.src_energy_usage
UNION ALL
SELECT 'device_metrics' AS stream, COUNT(*) AS rows FROM energy_catalog.raw.src_device_metrics
UNION ALL
SELECT 'grid_load'      AS stream, COUNT(*) AS rows FROM energy_catalog.raw.src_grid_load
UNION ALL
SELECT 'tariff_metrics' AS stream, COUNT(*) AS rows FROM energy_catalog.raw.src_tariff_metrics
UNION ALL
SELECT 'weather'        AS stream, COUNT(*) AS rows FROM energy_catalog.raw.src_weather;

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, lit, col
import datetime
import logging

# Logger

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze_all_streams")


# Global Configuration

BUCKET     = "s3://energy-pipline-prod"
batch_date = datetime.date.today().strftime("%Y-%m-%d")


# Stream Schemas


# energy_usage — 17 cols | timestamp format: yyyy-MM-dd HH:mm:ss
energy_usage_schema = StructType([
    StructField("household_id",          StringType(), True),
    StructField("region_name",           StringType(), True),
    StructField("city_name",             StringType(), True),
    StructField("meter_type",            StringType(), True),
    StructField("customer_category",     StringType(), True),
    StructField("grid_zone",             StringType(), True),
    StructField("voltage_reading",       DoubleType(), True),
    StructField("current_reading",       DoubleType(), True),
    StructField("active_power_kw",       DoubleType(), True),
    StructField("reactive_power_kvar",   DoubleType(), True),
    StructField("energy_usage_kwh",      DoubleType(), True),
    StructField("frequency_hz",          DoubleType(), True),
    StructField("load_factor",           DoubleType(), True),
    StructField("peak_demand_kw",        DoubleType(), True),
    StructField("offpeak_demand_kw",     DoubleType(), True),
    StructField("daily_consumption_kwh", DoubleType(), True),
    StructField("timestamp",             StringType(), True),  # yyyy-MM-dd HH:mm:ss — parse in Silver
])

# grid_load — 16 cols | no timestamp
grid_load_schema = StructType([
    StructField("household_id",      StringType(), True),
    StructField("grid_region",       StringType(), True),
    StructField("substation_name",   StringType(), True),
    StructField("feeder_line",       StringType(), True),
    StructField("distribution_zone", StringType(), True),
    StructField("grid_operator",     StringType(), True),
    StructField("grid_voltage",      DoubleType(), True),
    StructField("grid_current",      DoubleType(), True),
    StructField("grid_load_kw",      DoubleType(), True),
    StructField("transformer_load",  DoubleType(), True),
    StructField("line_loss_percent", DoubleType(), True),
    StructField("load_variation",    DoubleType(), True),
    StructField("frequency_variation", DoubleType(), True),
    StructField("grid_capacity_kw",  DoubleType(), True),
    StructField("demand_forecast_kw", DoubleType(), True),
    StructField("reserve_margin",    DoubleType(), True),
])

# tariff_metrics — 16 cols | no timestamp
# NOTE: adjustment_amount ranges -49.99 to +50.00 — negatives are billing CREDITS, not errors
tariff_metrics_schema = StructType([
    StructField("household_id",      StringType(), True),
    StructField("tariff_region",     StringType(), True),
    StructField("tariff_city",       StringType(), True),
    StructField("tariff_plan_type",  StringType(), True),
    StructField("billing_cycle",     StringType(), True),
    StructField("utility_provider",  StringType(), True),
    StructField("unit_rate",         DoubleType(), True),
    StructField("peak_rate",         DoubleType(), True),
    StructField("offpeak_rate",      DoubleType(), True),
    StructField("fixed_charge",      DoubleType(), True),
    StructField("tax_amount",        DoubleType(), True),
    StructField("subsidy_amount",    DoubleType(), True),
    StructField("monthly_bill",      DoubleType(), True),
    StructField("billing_units",     DoubleType(), True),
    StructField("late_fee",          DoubleType(), True),
    StructField("adjustment_amount", DoubleType(), True),
])

# device_metrics — 16 cols | no timestamp
device_metrics_schema = StructType([
    StructField("household_id",        StringType(), True),
    StructField("device_category",     StringType(), True),
    StructField("device_brand",        StringType(), True),
    StructField("device_model",        StringType(), True),
    StructField("maintenance_status",  StringType(), True),
    StructField("installation_region", StringType(), True),
    StructField("runtime_hours",       DoubleType(), True),
    StructField("device_power_kw",     DoubleType(), True),
    StructField("motor_speed_rpm",     DoubleType(), True),
    StructField("efficiency_ratio",    DoubleType(), True),
    StructField("energy_draw_kwh",     DoubleType(), True),
    StructField("heat_output",         DoubleType(), True),
    StructField("cooling_load",        DoubleType(), True),
    StructField("device_voltage",      DoubleType(), True),
    StructField("device_current",      DoubleType(), True),
    StructField("device_temperature",  DoubleType(), True),
])

# weather — 17 cols | timestamp format: dd-MM-yyyy (DIFFERENT from energy_usage)
weather_schema = StructType([
    StructField("household_id",        StringType(), True),
    StructField("weather_region",      StringType(), True),
    StructField("weather_city",        StringType(), True),
    StructField("weather_station",     StringType(), True),
    StructField("climate_zone",        StringType(), True),
    StructField("condition_type",      StringType(), True),
    StructField("temperature_celsius", DoubleType(), True),
    StructField("humidity_percent",    DoubleType(), True),
    StructField("wind_speed_kmh",      DoubleType(), True),
    StructField("rainfall_mm",         DoubleType(), True),
    StructField("pressure_hpa",        DoubleType(), True),
    StructField("solar_radiation",     DoubleType(), True),
    StructField("dew_point",           DoubleType(), True),
    StructField("uv_index",            DoubleType(), True),
    StructField("visibility_km",       DoubleType(), True),
    StructField("cloud_cover_percent", DoubleType(), True),
    StructField("timestamp",           StringType(), True),  # dd-MM-yyyy — parse in Silver
])


# Stream Registry
# Each entry: (stream_name, source_table, bronze_path, catalog_table, schema)

STREAMS = [
    (
        "energy_usage",
        "energy_catalog.raw.src_energy_usage",
        f"{BUCKET}/bronze/energy_usage/",
        "energy_catalog.raw.bronze_energy_usage",
        energy_usage_schema,
    ),
    (
        "grid_load",
        "energy_catalog.raw.src_grid_load",
        f"{BUCKET}/bronze/grid_load/",
        "energy_catalog.raw.bronze_grid_load",
        grid_load_schema,
    ),
    (
        "tariff_metrics",
        "energy_catalog.raw.src_tariff_metrics",
        f"{BUCKET}/bronze/tariff_metrics/",
        "energy_catalog.raw.bronze_tariff_metrics",
        tariff_metrics_schema,
    ),
    (
        "device_metrics",
        "energy_catalog.raw.src_device_metrics",
        f"{BUCKET}/bronze/device_metrics/",
        "energy_catalog.raw.bronze_device_metrics",
        device_metrics_schema,
    ),
    (
        "weather",
        "energy_catalog.raw.src_weather",
        f"{BUCKET}/bronze/weather/",
        "energy_catalog.raw.bronze_weather",
        weather_schema,
    ),
]

# ──────────────────────────────────────────────────
# Bronze Ingestion Function
# ──────────────────────────────────────────────────
def run_bronze_ingestion(stream_name, source_table, bronze_path, catalog_table, schema):
    """
    Ingests one stream from its source table into the Bronze Delta layer.
    Overwrites the existing table on each run — no duplicate accumulation.
    """
    logger.info("=" * 60)
    logger.info(f"Starting Bronze ingestion — stream : {stream_name}")
    logger.info(f"Batch date   : {batch_date}")
    logger.info(f"Source table : {source_table}")
    logger.info(f"Target table : {catalog_table}")
    logger.info("=" * 60)

    try:
        # ── Check source exists and has data ──────────────────────
        try:
            source_count = spark.sql(
                f"SELECT COUNT(*) AS cnt FROM {source_table}"
            ).collect()[0]["cnt"]
        except Exception:
            logger.warning(f"Source table not found: {source_table}")
            source_count = 0

        if source_count == 0:
            logger.warning(f"[{stream_name}] No records in source. Bronze ingestion skipped.")
            return

        logger.info(f"[{stream_name}] Records detected in source : {source_count}")

        # ── Load source data ──────────────────────────────────────
        df = spark.table(source_table)
        logger.info(f"[{stream_name}] Source data loaded")

        # ── Normalize column names to lowercase ───────────────────
        df = df.toDF(*[c.lower() for c in df.columns])
        logger.info(f"[{stream_name}] Column names normalized to lowercase")

        # ── Add Bronze audit metadata columns ─────────────────────
        df_bronze = (
            df
            .withColumn("_batch_date",   lit(batch_date))
            .withColumn("_ingestion_ts", current_timestamp())
            .withColumn("_source_file",  col("_metadata.file_path"))
            .withColumn("_stream_name",  lit(stream_name))
        )
        logger.info(f"[{stream_name}] Audit columns added — total columns : {len(df_bronze.columns)}")

        # ── Ensure catalog schema exists ──────────────────────────
        spark.sql("CREATE SCHEMA IF NOT EXISTS energy_catalog.raw")
        logger.info(f"[{stream_name}] Schema energy_catalog.raw verified")

        # ── Write to Bronze Delta (OVERWRITE) ─────────────────────
        # overwriteSchema=true is required when overwriting a partitioned table
        # to safely replace the existing data and schema on each run.
        df_bronze.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .partitionBy("_batch_date") \
            .save(bronze_path)
        logger.info(f"[{stream_name}] Delta overwrite complete → {bronze_path}")

        # ── Register table in Unity Catalog (idempotent) ──────────
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {catalog_table}
            USING DELTA
            LOCATION '{bronze_path}'
        """)
        logger.info(f"[{stream_name}] Table registered : {catalog_table}")

        # ── Optimize for query performance ────────────────────────
        spark.sql(f"OPTIMIZE {catalog_table} ZORDER BY (household_id)")
        logger.info(f"[{stream_name}] OPTIMIZE complete")

        # ── Verify final row count ────────────────────────────────
        written = spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {catalog_table}"
        ).collect()[0]["cnt"]
        logger.info(f"[{stream_name}] Rows in Bronze table after overwrite : {written}")

        # ── Null household_id check ───────────────────────────────
        null_hh = spark.sql(f"""
            SELECT COUNT(*) AS cnt FROM {catalog_table}
            WHERE household_id IS NULL
        """).collect()[0]["cnt"]

        if null_hh > 0:
            logger.warning(f"[{stream_name}] NULL household_ids found : {null_hh}")
        else:
            logger.info(f"[{stream_name}] NULL check passed — 0 null household_ids")

        logger.info(f"[{stream_name}] Bronze ingestion SUCCESSFUL ✓")

    except Exception as e:
        logger.error(f"[{stream_name}] Bronze ingestion FAILED")
        logger.error(str(e))
        raise  # Re-raise so Databricks Job marks the task as Failed

    finally:
        logger.info(f"[{stream_name}] Bronze pipeline execution finished")



# Main — Run all streams sequentially

if __name__ == "__main__" or "spark" in dir():

    failed_streams = []

    for stream_name, source_table, bronze_path, catalog_table, schema in STREAMS:
        try:
            run_bronze_ingestion(stream_name, source_table, bronze_path, catalog_table, schema)
        except Exception:
            # Log and continue — all streams attempted even if one fails
            failed_streams.append(stream_name)

    logger.info("=" * 60)
    if failed_streams:
        logger.error(f"Bronze pipeline completed with FAILURES: {failed_streams}")
        raise RuntimeError(f"The following streams failed: {failed_streams}")
    else:
        logger.info(f"All {len(STREAMS)} Bronze streams completed SUCCESSFULLY ✓")
    logger.info("=" * 60)